# 01 — Setup Unity Catalog & Synthetic Dummy Data: Real Estate (Property Listings, Market Valuation & AVM Pricing)

**Purpose:** Provision the Unity Catalog 3-level namespace (`real_estate_prod.bronze/silver/gold/ml`), managed Volumes for raw landing and Auto Loader checkpoints, and generate realistic synthetic dummy data.

**Prerequisites:** Unity Catalog-enabled Databricks workspace and a Databricks Runtime 15.4 LTS+ (or Serverless) cluster with `CREATE CATALOG` privileges.

In [ ]:
# Databricks Widget Parameters (overridden by Lakeflow Jobs / Databricks Asset Bundles)
dbutils.widgets.text("catalog_name", "real_estate_prod", "Unity Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_landing"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/bronze/checkpoints"
print(f"Active Catalog: {CATALOG} | Landing Volume: {VOLUME_PATH}")

## Step 1: Create Unity Catalog, Medallion Schemas & Managed Volumes

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'End-to-end real_estate Lakehouse catalog'")
spark.sql(f"USE CATALOG {CATALOG}")

for schema_name, comment in [
    ("bronze", "Raw ingested Delta tables and landing volumes"),
    ("silver", "Cleaned, deduplicated, and enriched Delta tables"),
    ("gold", "Business KPI aggregations, dashboard views, and batch ML predictions"),
    ("ml", "Unity Catalog feature tables and registered MLflow models"),
]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name} COMMENT '{comment}'")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw_landing COMMENT 'Raw JSON/CSV landing zone'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.checkpoints COMMENT 'Auto Loader schema & streaming checkpoints'")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## Step 2: Generate Realistic Synthetic Dummy Data into Unity Catalog Volume

In [ ]:
from pyspark.sql import functions as F

NUM_PROPERTIES = 20000
NUM_SALES = 12000

# 1. Property Listings (JSON)
props_df = (
    spark.range(0, NUM_PROPERTIES)
    .withColumn("property_id", F.concat(F.lit("PROP-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("city", F.element_at(F.array(F.lit("Austin"), F.lit("Seattle"), F.lit("Denver"), F.lit("Miami"), F.lit("Bengaluru")), (F.col("id") % 5 + 1).cast("int")))
    .withColumn("property_type", F.element_at(F.array(F.lit("SingleFamily"), F.lit("Condo"), F.lit("Townhouse"), F.lit("Villa")), (F.col("id") % 4 + 1).cast("int")))
    .withColumn("sqft", F.round(F.lit(650.0) + F.rand() * 3800.0, 0).cast("int"))
    .withColumn("bedrooms", (F.lit(1) + (F.col("sqft") / 850).cast("int")))
    .withColumn("bathrooms", F.round(F.lit(1.0) + (F.col("bedrooms") * 0.7), 1))
    .withColumn("year_built", (F.lit(1975) + (F.rand() * 50).cast("int")))
    .withColumn("school_rating", F.round(F.lit(3.0) + F.rand() * 7.0, 1))
    .withColumn("days_on_market", (F.rand() * 140).cast("int"))
    .withColumn("listing_date", F.expr("date_sub(current_date(), cast(rand() * 365 as int))"))
    .withColumn("list_price_usd", F.round(
        F.lit(85000.0)
        + F.col("sqft") * (F.when(F.col("city") == "Seattle", 410).when(F.col("city") == "Miami", 360).when(F.col("city") == "Austin", 310).otherwise(260))
        + F.col("school_rating") * 12000.0
        + (F.col("year_built") - 1975) * 900.0
        + (F.rand() - 0.5) * 45000.0, 0
    ))
    .drop("id")
)

props_path = f"{VOLUME_PATH}/property_listings"
props_df.write.mode("overwrite").json(props_path)
print(f"Wrote {props_df.count():,} property listings JSON records to {props_path}")

# 2. Closed Sales History (CSV)
sales_df = (
    spark.range(0, NUM_SALES)
    .withColumn("sale_id", F.concat(F.lit("SALE-"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("property_id", F.concat(F.lit("PROP-"), F.lpad((F.col("id") % NUM_PROPERTIES).cast("string"), 6, "0")))
    .withColumn("close_date", F.expr("date_sub(current_date(), cast(rand() * 180 as int))"))
    .withColumn("sale_to_list_ratio", F.round(F.lit(0.91) + F.rand() * 0.14, 3))
    .withColumn("buyer_financing", F.element_at(F.array(F.lit("CONVENTIONAL"), F.lit("CASH"), F.lit("FHA"), F.lit("JUMBO")), (F.col("id") % 4 + 1).cast("int")))
    .drop("id")
)

sales_path = f"{VOLUME_PATH}/property_sales_history"
sales_df.write.mode("overwrite").option("header", "true").csv(sales_path)
print(f"Wrote {sales_df.count():,} sales history CSV records to {sales_path}")

## Step 3: Verify Generated Landing Files in Unity Catalog Volume

In [ ]:
for folder in ["property_listings", "property_sales_history"]:
    files = dbutils.fs.ls(f"{VOLUME_PATH}/{folder}")
    print(f"{folder}: {len(files)} files written under {VOLUME_PATH}/{folder}")